# 第10章 pandas 入門 (DataFrame, merge, 欠損値処理)

データ分析の必須ライブラリ **pandas** の基本を学びます。
特にデータ分析の現場でほぼ毎回使う:

- `DataFrame` の作成・参照
- `pd.merge` による **複数の表の結合**
- `fillna` による **欠損値の補完**

を重点的に扱います。

## 学習目標
- `Series` と `DataFrame` の違いが分かる
- 列選択・条件抽出・集計ができる
- `pd.merge` の **inner / outer / left / right** を使い分けられる
- 欠損値の確認と補完 (`isnull`, `fillna`) ができる


## 10.1 import

In [ ]:
import pandas as pd
import numpy as np


## 10.2 Series (1 次元データ)

`Series` は **インデックスつきの 1 次元配列**。リストや辞書から作れます。


In [ ]:
s = pd.Series([10, 20, 30, 40])
print(s)
print('type:', type(s))


In [ ]:
# インデックスを指定して作る
s = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
print(s)
print(s['a'])      # ラベルで参照
print(s.iloc[0])   # 位置で参照


## 10.3 DataFrame (2 次元データ)

`DataFrame` は **表形式 (行 × 列)** のデータ。
辞書で作ると、キーが列名、値が列の中身になります。


In [ ]:
df = pd.DataFrame({
    'name': ['田中', '佐藤', '鈴木', '高橋'],
    'age':  [25, 30, 28, 35],
    'dept': ['営業', '開発', '営業', '開発'],
})
df


## 10.4 DataFrame の基本情報

In [ ]:
print('shape:', df.shape)       # (行数, 列数)
print('columns:', df.columns.tolist())
print('dtypes:')
print(df.dtypes)
print('\n--- head ---')
print(df.head(2))
print('\n--- describe (数値列のみ) ---')
print(df.describe())


## 10.5 列の選択


In [ ]:
# 1 列だけ取り出す → Series
print(df['name'])
print(type(df['name']))

# 複数列を取り出す → DataFrame
print(df[['name', 'age']])


## 10.6 行の選択

- `df.loc[ラベル]` → **ラベル** (インデックス値や列名) で指定
- `df.iloc[位置]` → **位置** (0 始まりの整数) で指定


In [ ]:
print(df.loc[0])      # インデックス 0 の行 (Series で返る)
print(df.iloc[0])     # 同じ (位置 0)

# 範囲
print(df.iloc[0:2])   # 先頭 2 行 (DataFrame)


## 10.7 条件抽出 (フィルタ)


In [ ]:
# 30 歳以上だけ
print(df[df['age'] >= 30])

# AND / OR は & / | を使う (and/or ではない！) しかも各条件は () で囲む
print(df[(df['age'] >= 28) & (df['dept'] == '開発')])


## 10.8 列の追加・更新・削除


In [ ]:
df['age_x2'] = df['age'] * 2   # 新しい列を追加
print(df)


In [ ]:
df['age'] = df['age'] + 1      # 既存列を更新
print(df)


In [ ]:
df = df.drop(columns=['age_x2'])   # 列を削除
print(df)


## 10.9 集計とグループ化


In [ ]:
print('平均年齢:', df['age'].mean())
print('合計:', df['age'].sum())
print('最大:', df['age'].max())
print('最小:', df['age'].min())
print('部署ごとの人数:')
print(df.groupby('dept').size())
print('部署ごとの平均年齢:')
print(df.groupby('dept')['age'].mean())


## 10.10 DataFrame の結合 (merge)

異なる DataFrame を **共通のキー列** でつなげる操作を「結合 (join)」と言います。
SQL の `JOIN` と同じ考え方で、`pd.merge` で行います。

### 構文

```python
pd.merge(左DF, 右DF, on='キー列', how='結合方法')
```

### 4 種類の `how` 引数

| how | 意味 | 残るデータ |
|-|-|-|
| `inner` | 内部結合 (デフォルト) | **両方に存在するキー** のみ |
| `outer` | 完全外部結合 | **どちらか一方にでもあるキー** すべて (足りない値は NaN) |
| `left` | 左外部結合 | 左の DataFrame のキーをすべて残す (右に無いと NaN) |
| `right` | 右外部結合 | 右の DataFrame のキーをすべて残す (左に無いと NaN) |


In [ ]:
df_emp = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'name':   ['Alice', 'Bob', 'Carol', 'Dave'],
})

df_dept = pd.DataFrame({
    'emp_id':   [1, 2, 5],          # Carol(3), Dave(4) は登録なし、5 は社員一覧にない
    'dept':     ['Sales', 'Dev', 'HR'],
})

print('df_emp')
print(df_emp)
print()
print('df_dept')
print(df_dept)


### inner: 両方に存在するキーのみ

In [ ]:
pd.merge(df_emp, df_dept, on='emp_id', how='inner')
# emp_id=1 (Alice/Sales), emp_id=2 (Bob/Dev) の 2 行だけが残る


### outer: 両方の和集合 (足りない値は NaN)

In [ ]:
pd.merge(df_emp, df_dept, on='emp_id', how='outer')
# emp_id=3,4 は df_dept に無いので dept が NaN
# emp_id=5 は df_emp に無いので name が NaN


### left: 左の DataFrame をすべて残す

In [ ]:
pd.merge(df_emp, df_dept, on='emp_id', how='left')
# emp_id=1,2,3,4 が残る。3,4 の dept は NaN


### right: 右の DataFrame をすべて残す

In [ ]:
pd.merge(df_emp, df_dept, on='emp_id', how='right')
# emp_id=1,2,5 が残る。5 の name は NaN


### 4 つを並べて比較

In [ ]:
for how in ['inner', 'outer', 'left', 'right']:
    print(f'\n=== how={how!r} ===')
    print(pd.merge(df_emp, df_dept, on='emp_id', how=how))


## 10.11 欠損値 (NaN) の確認と処理

pandas では欠損値は `NaN` (Not a Number) で表され、`np.nan` で明示的に作れます。


In [ ]:
df = pd.DataFrame({
    'A': [1.0, 2.0, np.nan, 4.0, np.nan, 6.0],
    'B': ['x', np.nan, 'z', 'w', 'v', np.nan],
})
df


In [ ]:
# 欠損の有無を確認
print(df.isnull())
print('\n列ごとの欠損数:')
print(df.isnull().sum())


### fillna の主な方法

| 方法 | 説明 |
|-|-|
| `fillna(値)` | 全欠損を指定値で埋める |
| `fillna({'col1': 値1, 'col2': 値2})` | 列ごとに違う値で埋める |
| `fillna(df.mean())` | 列ごとの平均で埋める (数値列のみ可) |
| `fillna(method='ffill')` | **前(上)の有効値で埋める** (forward fill) |
| `fillna(method='bfill')` | **後(下)の有効値で埋める** (backward fill) |


In [ ]:
# 一律 0 で埋める
df.fillna(0)


In [ ]:
# 列ごとに違う値で埋める
df.fillna({'A': df['A'].mean(), 'B': '(なし)'})


In [ ]:
# ffill: 前 (上) の値で埋める
df['A'].fillna(method='ffill')


In [ ]:
# bfill: 後 (下) の値で埋める
df['A'].fillna(method='bfill')


### dropna: 欠損を含む行/列を削除

In [ ]:
# 欠損を含む行を削除 (axis=0 がデフォルト)
df.dropna()


In [ ]:
# 欠損を含む列を削除
df.dropna(axis=1)


## 10.12 CSV の読み書き


In [ ]:
# CSV に書き出し
df.to_csv('/tmp/example.csv', index=False)

# CSV から読み込み
df_loaded = pd.read_csv('/tmp/example.csv')
df_loaded


## 10.13 練習問題


In [ ]:
# Q1. inner join の結果は?
df_a = pd.DataFrame({'id': [1, 2, 3], 'val_a': [10, 20, 30]})
df_b = pd.DataFrame({'id': [2, 3, 4], 'val_b': [200, 300, 400]})

pd.merge(df_a, df_b, on='id', how='inner')


**Q1 解説**

両方に存在する `id` は 2 と 3 のみ。
よって 2 行残り、`val_a` と `val_b` が並ぶ。


In [ ]:
# Q2. ffill の結果, インデックス 3 の col の値は?
df = pd.DataFrame({
    'col': [1.0, np.nan, 3.0, np.nan, 5.0],
})
print(df)
print('\n--- ffill ---')
print(df.fillna(method='ffill'))


**Q2 解説**

| index | 元 | ffill 後 |
|-|-|-|
| 0 | 1.0 | 1.0 |
| 1 | NaN | **1.0** (前の値) |
| 2 | 3.0 | 3.0 |
| 3 | NaN | **3.0** (前の値) |
| 4 | 5.0 | 5.0 |

インデックス 3 は **3.0**。


In [ ]:
# Q3. bfill だとどうなる?
print(df.fillna(method='bfill'))


**Q3 解説**

| index | 元 | bfill 後 |
|-|-|-|
| 0 | 1.0 | 1.0 |
| 1 | NaN | **3.0** (次の有効値) |
| 2 | 3.0 | 3.0 |
| 3 | NaN | **5.0** (次の有効値) |
| 4 | 5.0 | 5.0 |


## まとめ

- `Series` は 1 次元, `DataFrame` は 2 次元のデータ構造
- 列選択は `df['col']`, 行選択は `df.loc / df.iloc`
- 条件抽出は `&` `|` を使い、各条件を `()` で囲む
- `pd.merge(左, 右, on=..., how=...)` で結合 (4 種類の how)
- 欠損値は `isnull` で確認, `fillna` / `dropna` で処理
- `ffill` は前の値, `bfill` は後の値で埋める
